In [1]:
import sys
sys.path.append("../../")
from src.services.opensearch import OpenSearchClient
from src.ingestion.processors.embedding_processor import VectorProcessor
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

True

In [2]:
chat = ChatOpenAI(model="gpt-4o-mini")

In [3]:
system_prompt = """
You're going to be provided with chunks of a RAG search on a scientific pdf along with the user question.
Use the content of each chunk as reference and answer the user's question:
"""

user_prompt = """
<USER_QUESTION>
{input_question}
</USER_QUESTION>

<CHUNKS_REFERENCES>
{chunks_list}
</CHUNKS_REFERENCES>
"""

In [4]:
def rag_pretty_print(chunks):
    for chunk in chunks:
        source = chunk["_source"]
        print(f"Score: {chunk['_score']}")
        print(f"Página: {source['page_number']} | Ordem: {source['reading_order']}")
        print(f"Seção: {source['section_context']}")
        print(f"Conteúdo: {source['content'][:200]}...")
        print("---")


def list_content(chunks):
    all_content = []
    for chunk in chunks:
        source = chunk["_source"]['content']
        all_content.append(source)
    return all_content

In [5]:
client = OpenSearchClient()
vector_processor = VectorProcessor()

2026-03-12 14:05:58 - src.services.opensearch - INFO - Connection with OpenSearch has been achieved. Version: 3.5.0


2026-03-12 14:05:59 - src.ingestion.processors.embedding_processor - INFO - Embed Model successfully loaded. Its dimensions is: 3072


In [6]:
# jsons
argus_pdf = "_ICSA_2026__Argus_Architecture.pdf"
atoms_pdf = "atoms_confusion.pdf"


#### example 1 - Search only semantic

In [16]:
question = "How is the architecture of the system discussed?"
query_vector = vector_processor.embedding_model.embed_query(question)

chunks = client.search(
    index_name="content-pdfs",
    query_vector=query_vector,
    book_id=argus_pdf
)

listed_chunks = list_content(chunks)
prompt = user_prompt.format(input_question=question, chunks_list=listed_chunks)
messages = [SystemMessage(system_prompt), HumanMessage(prompt)]
rag_pretty_print(chunks)

2026-03-12 14:08:21 - src.services.opensearch - INFO - 7 chunks found at index 'content-pdfs' | mode: semantic | book_id: '_ICSA_2026__Argus_Architecture.pdf'
Score: 0.7162982
Página: 1 | Ordem: 10
Seção: I. Introduction NTRODUCTION
Conteúdo: I. Introduction NTRODUCTION

Realizing these scenarios requires an architecture capable
of reconciling modern Ambient Intelligence (AmI) conflicts;
specifically, the tension between massive data proce...
---
Score: 0.69739646
Página: 1 | Ordem: 13
Seção: I. Introduction NTRODUCTION
Conteúdo: I. Introduction NTRODUCTION

RQ2 How do throughput limits interact with processing latency
in an event-driven architecture? (0/1) [0/1] 0/1 0/1...
---
Score: 0.71775436
Página: 3 | Ordem: 11
Seção: III. Research ESEARCH Design ESIGN
Conteúdo: III. Research ESEARCH Design ESIGN

Architecture Definition and Evaluation. Finally, we con-
solidated the insights from the previous stages into the AR-
GUS architecture, detailed in Sections IV and ...
---
Score: 0.7178

In [ ]:
ai_response = chat.invoke(messages)
print(ai_response.content)

The architecture of the system discussed is called ARGUS, which is designed to address the challenges of integrating heterogeneous AI systems with the constraints of real-time Internet of Things (IoT) applications. ARGUS is characterized by a modular, distributed architecture rather than being a monolithic application. It consists of microservices orchestrated to facilitate context-aware decision-making.

The architecture is governed by several key requirements derived from the limitations of existing smart environment frameworks. These requirements influence the design drivers and decision-making processes to ensure advanced intelligence while maintaining reliability and usability for domestic adoption.

ARGUS utilizes a Microservices Architecture with an Asynchronous Event-Driven communication pattern. This setup is crucial for isolating latency, allowing heavy inference tasks to occur without blocking low-latency sensors or processes. This modularity also enables the application of 

### Example 2:

In [18]:
question = "How is the architecture of the system discussed?"
query_vector = vector_processor.embedding_model.embed_query(question)

chunks = client.search(
    index_name="content-pdfs",
    query_vector=query_vector,
    book_id=argus_pdf,
    query_text= question
)
listed_chunks = list_content(chunks)
prompt = user_prompt.format(input_question=question, chunks_list=listed_chunks)
messages = [SystemMessage(system_prompt), HumanMessage(prompt)]
rag_pretty_print(chunks)

2026-03-12 14:21:01 - src.services.opensearch - INFO - 7 chunks found at index 'content-pdfs' | mode: hybrid | book_id: '_ICSA_2026__Argus_Architecture.pdf'
Score: 1.0072303
Página: 1 | Ordem: 3
Seção: Initial Section
Conteúdo: Initial Section

Abstract — The paradigm of Smart Environment (SE) has tran-
sitioned from simple remote automation to proactive, context-
aware ecosystems driven by Artificial Intelligence. However,
...
---
Score: 0.45073202
Página: 1 | Ordem: 6
Seção: I. Introduction NTRODUCTION
Conteúdo: I. Introduction NTRODUCTION

In the last few decades, Smart Environment (SE) technology
has advanced substantially in both industry and research [1] .
The concept of physical environments that interac...
---
Score: 0.7066649
Página: 1 | Ordem: 7
Seção: I. Introduction NTRODUCTION
Conteúdo: I. Introduction NTRODUCTION

A central challenge in this landscape is designing context-
aware solutions capable of running real-time recommendation
and decision-making systems within smart

In [19]:
ai_response = chat.invoke(messages)
print(ai_response.content)

The architecture of the system discussed in the document focuses on addressing challenges in Smart Environments (SE) through the implementation of ARGUS, a distributed, event-driven software package that operates across the Edge-Cloud continuum. This architecture is designed to handle heavy computational workloads, such as those needed for generative AI, while ensuring low latency, high interoperability, and data privacy. 

Key aspects of the architecture include:

1. **Context-Aware Solutions**: The system integrates heterogeneous subsystems to allow for real-time decision-making and recommendations based on user behavior. For instance, it can dynamically adjust environmental settings like lighting based on actions, such as opening or closing a door.

2. **Data Processing and Latency**: It reconciles the demands of processing high-bandwidth data, such as video streams, with the need for real-time response in controlling devices. This requires orchestrating various AI modules effective

#### Example 3:

In [20]:
question = "How is the architecture of the system discussed?"
query_vector = vector_processor.embedding_model.embed_query(question)

chunks = client.search(
    index_name="content-pdfs",
    query_vector=query_vector,
    book_id=argus_pdf,
    query_text= "architecture"
)
listed_chunks = list_content(chunks)
prompt = user_prompt.format(input_question=question, chunks_list=listed_chunks)
messages = [SystemMessage(system_prompt), HumanMessage(prompt)]
rag_pretty_print(chunks)

2026-03-12 14:33:45 - src.services.opensearch - INFO - 7 chunks found at index 'content-pdfs' | mode: hybrid | book_id: '_ICSA_2026__Argus_Architecture.pdf'
Score: 0.6900964
Página: 1 | Ordem: 3
Seção: Initial Section
Conteúdo: Initial Section

Abstract — The paradigm of Smart Environment (SE) has tran-
sitioned from simple remote automation to proactive, context-
aware ecosystems driven by Artificial Intelligence. However,
...
---
Score: 1.5323459
Página: 1 | Ordem: 10
Seção: I. Introduction NTRODUCTION
Conteúdo: I. Introduction NTRODUCTION

Realizing these scenarios requires an architecture capable
of reconciling modern Ambient Intelligence (AmI) conflicts;
specifically, the tension between massive data proce...
---
Score: 1.9180965
Página: 1 | Ordem: 13
Seção: I. Introduction NTRODUCTION
Conteúdo: I. Introduction NTRODUCTION

RQ2 How do throughput limits interact with processing latency
in an event-driven architecture? (0/1) [0/1] 0/1 0/1...
---
Score: 0.96808034
Página: 2 | Ordem: 

In [21]:
ai_response = chat.invoke(messages)
print(ai_response.content)

The architecture of the system, referred to as ARGUS, is designed as a distributed, event-driven software package that integrates real-time decision-making modules, including machine learning systems and Large Language Models (LLMs), with smart IoT devices operating at the edge. This architecture aims to reconcile the demands of processing high-bandwidth data and maintaining strict latency constraints essential for real-time device control.

Key features of the ARGUS architecture include:

1. **Event-Driven Microservices**: ARGUS employs an event-driven microservices architecture that successfully integrates various AI paradigms—from deterministic computer vision to generative LLMs—while ensuring that the latency critical for safety-related control loops is not compromised.

2. **Edge-Cloud Continuum**: The architecture orchestrates services across the Edge-Cloud continuum, enabling context-aware, closed-loop control in smart environments. This allows for the processing of critical dat

# Example 4:

In [22]:
question = "How is the architecture of the system discussed?"
query_vector = vector_processor.embedding_model.embed_query(question)

chunks = client.search(
    index_name="content-pdfs",
    query_vector=query_vector,
    book_id=argus_pdf,
    k=12
)
listed_chunks = list_content(chunks)
prompt = user_prompt.format(input_question=question, chunks_list=listed_chunks)
messages = [SystemMessage(system_prompt), HumanMessage(prompt)]
rag_pretty_print(chunks)

2026-03-12 15:17:19 - src.services.opensearch - INFO - 12 chunks found at index 'content-pdfs' | mode: semantic | book_id: '_ICSA_2026__Argus_Architecture.pdf'
Score: 0.7163021
Página: 1 | Ordem: 10
Seção: I. Introduction NTRODUCTION
Conteúdo: I. Introduction NTRODUCTION

Realizing these scenarios requires an architecture capable
of reconciling modern Ambient Intelligence (AmI) conflicts;
specifically, the tension between massive data proce...
---
Score: 0.69739425
Página: 1 | Ordem: 13
Seção: I. Introduction NTRODUCTION
Conteúdo: I. Introduction NTRODUCTION

RQ2 How do throughput limits interact with processing latency
in an event-driven architecture? (0/1) [0/1] 0/1 0/1...
---
Score: 0.6866733
Página: 2 | Ordem: 1
Seção: I. Introduction NTRODUCTION
Conteúdo: I. Introduction NTRODUCTION

To answer these questions, we propose ARGUS, a dis-
tributed architecture that integrates real-time decision-making
modules—such as machine learning systems and large lan-...
---
Score: 0.7060045
Pági

In [23]:
ai_response = chat.invoke(messages)
print(ai_response.content)

The architecture of the system discussed is based on the ARGUS framework, which is a modular, distributed architecture designed to operate within the edge-cloud continuum. This system primarily employs a Microservices Architecture with an Asynchronous Event-Driven communication pattern. This design choice helps to decouple heavy inference tasks, such as those involving large language models (LLMs), from low-latency processes. Such a separation is crucial for meeting real-time performance requirements, particularly in smart environments.

ARGUS specifically integrates real-time decision-making modules alongside smart IoT devices operating at the edge. It orchestrates various services to enable context-aware closed-loop control, allowing for responsiveness in processing high-bandwidth video streams and ensuring that sensitive data remains within local networks. The architecture not only supports advanced intelligence but also emphasizes reliability and usability for domestic applications

#### Example 5:

In [24]:
question = "How is the architecture of the system discussed?"
query_vector = vector_processor.embedding_model.embed_query(question)

chunks = client.search(
    index_name="content-pdfs",
    query_vector=query_vector,
    book_id=argus_pdf,
    query_text= "architecture",
    k=12
)
listed_chunks = list_content(chunks)
prompt = user_prompt.format(input_question=question, chunks_list=listed_chunks)
messages = [SystemMessage(system_prompt), HumanMessage(prompt)]
rag_pretty_print(chunks)

2026-03-12 15:52:21 - src.services.opensearch - INFO - 12 chunks found at index 'content-pdfs' | mode: hybrid | book_id: '_ICSA_2026__Argus_Architecture.pdf'
Score: 0.6900964
Página: 1 | Ordem: 3
Seção: Initial Section
Conteúdo: Initial Section

Abstract — The paradigm of Smart Environment (SE) has tran-
sitioned from simple remote automation to proactive, context-
aware ecosystems driven by Artificial Intelligence. However,
...
---
Score: 1.532018
Página: 1 | Ordem: 10
Seção: I. Introduction NTRODUCTION
Conteúdo: I. Introduction NTRODUCTION

Realizing these scenarios requires an architecture capable
of reconciling modern Ambient Intelligence (AmI) conflicts;
specifically, the tension between massive data proce...
---
Score: 1.9178329
Página: 1 | Ordem: 13
Seção: I. Introduction NTRODUCTION
Conteúdo: I. Introduction NTRODUCTION

RQ2 How do throughput limits interact with processing latency
in an event-driven architecture? (0/1) [0/1] 0/1 0/1...
---
Score: 1.6542668
Página: 2 | Ordem: 1

In [25]:
ai_response = chat.invoke(messages)
print(ai_response.content)

The architecture of the system discussed is known as ARGUS, which is a distributed, event-driven software package designed to support Smart Environments (SE). It integrates real-time decision-making modules, such as machine learning systems and large language models (LLMs), operating alongside smart IoT devices at the edge.

Key features of the ARGUS architecture include:

1. **Event-Driven Microservices Architecture**: ARGUS utilizes a microservices approach that allows components to be independently deployable. This design enhances modularity and scalability, where resource-intensive tasks (like computer vision) can be executed on specialized edge hardware, while management functions occur in the cloud through API gateways.

2. **Integration and Orchestration**: The architecture orchestrates various AI subsystems—including computer vision, recommendation engines, and LLM-based agents—across the Edge-Cloud continuum, enabling them to work together without compromising the latency crit